# Tomato Leaf Disease — CNN Model Benchmark


## 1. Setup and imports

In [ ]:
import os, time, random, itertools, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dropout, Dense,
                                      GlobalAveragePooling2D, Input)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.metrics import Precision, Recall, AUC, TopKCategoricalAccuracy
from tensorflow.keras.utils import to_categorical, normalize

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, roc_curve, auc as sk_auc)

import warnings
warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)

In [ ]:
# ----------------------------- Configuration -------------------------------
SEED = 123
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Dataset paths (PlantVillage "New Plant Diseases Dataset (Augmented)" on Kaggle).
# Split policy (proper 3-way):
#   train/  ->  TRAIN (1 - VAL_SPLIT)  +  VALIDATION (VAL_SPLIT), stratified
#   valid/  ->  TEST  (held out; used ONLY for the final reported metrics)
TRAIN_DIR = './new-plant-diseases-dataset/train/'
VALID_DIR = './new-plant-diseases-dataset/valid/'

# Training protocol (kept identical across all models, per the paper).
BATCH_SIZE   = 16
MAX_EPOCHS   = 100
LR           = 0.001
PATIENCE     = 5          # early-stopping patience on val_loss
VAL_SPLIT    = 0.2        # fraction of train/ held out as the validation split

# 10 tomato classes (folders contain the substring 'Tomato__').
CLASS_NAMES = [
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy',
]
NUM_CLASSES = len(CLASS_NAMES)

# Where trained models / result tables are written.
OUT_DIR = 'benchmark_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Data loading

`prepare_data(img_size)` builds a **proper three-way split** for any image size:

* **train/** is loaded and split (stratified) into a **training** set and a **validation** set (`VAL_SPLIT`, default 20%).
* **valid/** is loaded and kept as a **held-out test set** — it is *never* used for training or early-stopping, only for the final reported metrics.

Returns `(X_train, y_train, X_val, y_val, X_test, y_test)`.

In [ ]:
def load_data(dir_path, img_size=(128, 128)):
    """Load every 'Tomato__' subfolder under dir_path into arrays."""
    X, Y, labels, i = [], [], {}, 0
    for path in tqdm(sorted(os.listdir(dir_path)), desc=os.path.basename(dir_path.rstrip('/'))):
        if path.startswith('.') or 'Tomato__' not in path:
            continue
        labels[i] = path
        sub_dir = os.path.join(dir_path, path)
        for file in os.listdir(sub_dir):
            if file.startswith('.'):
                continue
            fp = os.path.join(sub_dir, file)
            if os.path.isfile(fp):
                img = cv2.imread(fp)
                img = Image.fromarray(img, 'RGB').resize(img_size)
                X.append(np.array(img)); Y.append(i)
        i += 1
    X = np.array(X); Y = np.array(Y)
    print(f'{len(X)} images loaded from {dir_path}')
    return X, Y, labels

In [ ]:
def prepare_data(img_size, num_classes=NUM_CLASSES, val_split=None):
    """Build a proper train / validation / test split.

    * train/ folder -> stratified split into TRAIN (1 - val_split) + VALIDATION (val_split)
    * valid/ folder -> held-out TEST set (never seen during training or model selection)
    """
    val_split = VAL_SPLIT if val_split is None else val_split
    size = (img_size, img_size) if isinstance(img_size, int) else img_size

    # Training pool (train/) and the untouched held-out test set (valid/).
    X_pool, y_pool, _ = load_data(TRAIN_DIR, size)
    X_test, y_test, _ = load_data(VALID_DIR, size)

    # Carve a stratified validation split out of the training pool.
    X_train, X_val, y_train, y_val = train_test_split(
        X_pool, y_pool, test_size=val_split, stratify=y_pool, random_state=SEED)

    # Shuffle every split.
    X_train, y_train = shuffle(X_train, y_train, random_state=101)
    X_val,   y_val   = shuffle(X_val,   y_val,   random_state=101)
    X_test,  y_test  = shuffle(X_test,  y_test,  random_state=101)

    # Normalise + one-hot encode each split.
    def _prep(X, y):
        X = normalize(X.astype(np.float32), axis=1)
        return X, to_categorical(y, num_classes=num_classes)
    X_train, y_train = _prep(X_train, y_train)
    X_val,   y_val   = _prep(X_val,   y_val)
    X_test,  y_test  = _prep(X_test,  y_test)

    print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test (held-out): {X_test.shape}')
    return X_train, y_train, X_val, y_val, X_test, y_test


## 3. Evaluation utilities

Defined once and reused for every model: confusion matrix, training-curve plots and a multi-class ROC curve (replaces the old external `Evaluation` import).

In [ ]:
def plot_confusion_matrix(cm, classes, normalize=True, title='Confusion Matrix',
                          cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right'); plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'; thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt), ha='center',
                 color='white' if cm[i, j] > thresh else 'black')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.tight_layout(); plt.show()


def plot_training_metrics(history):
    """Plot accuracy/loss/precision/recall/AUC/top-k curves if present."""
    h = history.history
    panels = [('accuracy', 'val_accuracy', 'Accuracy'), ('loss', 'val_loss', 'Loss'),
              ('precision', 'val_precision', 'Precision'), ('recall', 'val_recall', 'Recall'),
              ('auc', 'val_auc', 'AUC'),
              ('top_k_categorical_accuracy', 'val_top_k_categorical_accuracy', 'Top-K Acc')]
    for tr, va, name in panels:
        if tr not in h:
            continue
        plt.figure(figsize=(10, 4))
        plt.plot(h[tr], 'r', label=f'Training {name}')
        if va in h:
            plt.plot(h[va], 'b', label=f'Validation {name}')
        plt.title(name); plt.xlabel('Epoch'); plt.legend(); plt.show()


def plot_roc_curve(model, X_test, y_test):
    """Per-class one-vs-rest ROC curve with AUC in the legend."""
    y_score = model.predict(X_test, verbose=0)
    n_classes = y_test.shape[1]
    plt.figure(figsize=(8, 8))
    for c in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test[:, c], y_score[:, c])
        plt.plot(fpr, tpr, label=f'class {c} (AUC = {sk_auc(fpr, tpr):.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC Curve'); plt.legend(loc='lower right'); plt.show()

## 4. Models

A single `build_simplenet(...)` factory generates the whole SimpleNet family; the
pretrained backbones each have a small builder. Every model is compiled with the
**same** optimiser, loss and metrics so the benchmark is apples-to-apples.

**SimpleNet family (built from scratch)**

| Name | Conv filters | 3rd-conv | Dropout |
|------|--------------|----------|---------|
| SimpleNet-1 | 16, 16 | – | no |
| SimpleNet-2 | 32, 32 | – | no |
| SimpleNet-3 | 64, 64 | – | no |
| SimpleNetAug-1 | 16, 16, 16 | yes | no |
| SimpleNetAug-2 | 32, 32, 64 | yes | no |
| SimpleNetAug-3 | 64, 64, 64 | yes | no |
| SimpleNetAug-4 | 64, 64, 128 | yes | no |
| SimpleNetAugDR-3 | 64, 64, 64 | yes | 0.5 |
| SimpleNetAugDR-4 | 64, 64, 128 | yes | 0.5 |

First two conv layers use a 4×4 kernel; the augmented 3rd conv uses 3×3 (matching
the reference implementation). Each conv is followed by 2×2 max-pooling, then
Flatten → (Dropout) → Dense(softmax).

In [ ]:
def compile_model(model, lr=LR, num_classes=NUM_CLASSES):
    """Shared compilation. Explicit metric names avoid duplicated history keys
    (precision_1, recall_1, ...) when several models are built in one session."""
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=lr),
        metrics=['accuracy',
                 Precision(name='precision'),
                 Recall(name='recall'),
                 AUC(name='auc'),
                 TopKCategoricalAccuracy(k=3, name='top_k_categorical_accuracy')])
    return model


def build_simplenet(filters, dropout=None, input_size=128, num_classes=NUM_CLASSES):
    """SimpleNet / SimpleNetAug / SimpleNetAugDR factory.

    filters : list of conv filter counts (len 2 -> base, len 3 -> augmented).
    dropout : float or None (dropout before the dense output layer).
    """
    keras.backend.clear_session()
    m = Sequential(name='simplenet')
    m.add(Input(shape=(input_size, input_size, 3)))
    for idx, f in enumerate(filters):
        kernel = (4, 4) if idx < 2 else (3, 3)   # 3rd (augmented) conv uses 3x3
        m.add(Conv2D(f, kernel, activation='relu'))
        m.add(MaxPooling2D(pool_size=(2, 2)))
    m.add(Flatten())
    if dropout:
        m.add(Dropout(dropout))
    m.add(Dense(num_classes, activation='softmax'))
    return compile_model(m, num_classes=num_classes)


# Explicit configuration of every SimpleNet-family member.
SIMPLENET_CONFIGS = {
    'SimpleNet-1':      dict(filters=[16, 16],       dropout=None),
    'SimpleNet-2':      dict(filters=[32, 32],       dropout=None),
    'SimpleNet-3':      dict(filters=[64, 64],       dropout=None),
    'SimpleNetAug-1':   dict(filters=[16, 16, 16],   dropout=None),
    'SimpleNetAug-2':   dict(filters=[32, 32, 64],   dropout=None),
    'SimpleNetAug-3':   dict(filters=[64, 64, 64],   dropout=None),
    'SimpleNetAug-4':   dict(filters=[64, 64, 128],  dropout=None),
    'SimpleNetAugDR-3': dict(filters=[64, 64, 64],   dropout=0.5),
    'SimpleNetAugDR-4': dict(filters=[64, 64, 128],  dropout=0.5),
}

**Pretrained backbones (transfer learning)**

* **MobileNet1** — frozen MobileNet base → GAP → Dense(softmax).
* **MobileNet2** — frozen MobileNet base → GAP → Dense(256, relu) → Dense(softmax).
* **VGG-19 / EfficientNetB0 / ResNet50 / ResNet101 / DenseNet121 / InceptionV3** —
  ImageNet base fine-tuned, custom head GAP → Dense(256, relu) → Dense(softmax).

In [ ]:
from tensorflow.keras.applications import (MobileNet, VGG19, EfficientNetB0,
                                           ResNet50, ResNet101, DenseNet121, InceptionV3)

def _transfer_model(base, extra_dense=256, trainable=True,
                    input_size=128, num_classes=NUM_CLASSES):
    keras.backend.clear_session()
    base.trainable = trainable
    layers = [base, GlobalAveragePooling2D()]
    if extra_dense:
        layers.append(Dense(extra_dense, activation='relu'))
    layers.append(Dense(num_classes, activation='softmax'))
    return compile_model(Sequential(layers), num_classes=num_classes)

def build_mobilenet1(input_size=128, num_classes=NUM_CLASSES):
    base = MobileNet(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=None, trainable=False, input_size=input_size, num_classes=num_classes)

def build_mobilenet2(input_size=128, num_classes=NUM_CLASSES):
    base = MobileNet(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=256, trainable=False, input_size=input_size, num_classes=num_classes)

def build_vgg19(input_size=128, num_classes=NUM_CLASSES):
    base = VGG19(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=256, trainable=True, input_size=input_size, num_classes=num_classes)

def build_efficientnetb0(input_size=128, num_classes=NUM_CLASSES):
    base = EfficientNetB0(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=256, trainable=True, input_size=input_size, num_classes=num_classes)

def build_resnet50(input_size=128, num_classes=NUM_CLASSES):
    base = ResNet50(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=256, trainable=True, input_size=input_size, num_classes=num_classes)

def build_resnet101(input_size=128, num_classes=NUM_CLASSES):
    base = ResNet101(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=256, trainable=True, input_size=input_size, num_classes=num_classes)

def build_densenet121(input_size=128, num_classes=NUM_CLASSES):
    base = DenseNet121(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=256, trainable=True, input_size=input_size, num_classes=num_classes)

def build_inceptionv3(input_size=128, num_classes=NUM_CLASSES):
    base = InceptionV3(input_shape=(input_size, input_size, 3), include_top=False, weights='imagenet')
    return _transfer_model(base, extra_dense=256, trainable=True, input_size=input_size, num_classes=num_classes)

In [ ]:
# Single registry mapping a model name -> a zero-arg-ish builder(input_size, num_classes).
def _simplenet_builder(cfg):
    return lambda input_size=128, num_classes=NUM_CLASSES: build_simplenet(
        cfg['filters'], cfg['dropout'], input_size=input_size, num_classes=num_classes)

MODEL_REGISTRY = {name: _simplenet_builder(cfg) for name, cfg in SIMPLENET_CONFIGS.items()}
MODEL_REGISTRY.update({
    'MobileNet1':     build_mobilenet1,
    'MobileNet2':     build_mobilenet2,
    'EfficientNetB0': build_efficientnetb0,
    'VGG-19':         build_vgg19,
    'DenseNet121':    build_densenet121,
    'InceptionV3':    build_inceptionv3,
    'ResNet50':       build_resnet50,
    'ResNet101':      build_resnet101,
})

# The exact model order used in the paper's Table II.
TABLE2_ORDER = [
    'SimpleNet-1', 'SimpleNet-2', 'SimpleNet-3',
    'SimpleNetAug-1', 'SimpleNetAug-2', 'SimpleNetAug-3', 'SimpleNetAug-4',
    'SimpleNetAugDR-3', 'SimpleNetAugDR-4',
    'MobileNet1', 'MobileNet2', 'EfficientNetB0', 'VGG-19',
    'DenseNet121', 'InceptionV3', 'ResNet50', 'ResNet101',
]
TABLE4_ORDER = ['SimpleNetAugDR-3', 'MobileNet2']   # evaluated at 64x64
print(f'{len(MODEL_REGISTRY)} models registered.')

## 5. Benchmark harness

`benchmark_model` trains one model and measures every Table II/IV column:
accuracy, precision, recall (weighted), training time, **per-image** inference
time (ms), total parameters and on-disk model size (MB). `run_benchmark`
loops over a list of model names and returns a tidy DataFrame.

In [ ]:
def format_seconds(s):
    m, sec = divmod(int(round(s)), 60)
    return f'{m}min {sec}s' if m else f'{sec}s'


def measure_inference_time(model, X, n_warmup=2, n_runs=5):
    """Median wall-clock inference time per image, in milliseconds."""
    _ = model.predict(X[:BATCH_SIZE], batch_size=BATCH_SIZE, verbose=0)  # warmup graph
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(X, batch_size=BATCH_SIZE, verbose=0)
        times.append((time.perf_counter() - t0) / len(X) * 1000.0)
    return float(np.median(times))


def benchmark_model(name, builder, X_train, y_train, X_val, y_val, X_test, y_test,
                    input_size=128, verbose=1):
    """Train one model and evaluate it on the HELD-OUT TEST set.

    Training is monitored on the VALIDATION split (early stopping + best-weight
    restoration); the reported metrics are computed on the test set, which the
    model never saw during training or model selection.
    """
    print(f'\n{"="*70}\n{name}  (input {input_size}x{input_size})\n{"="*70}')
    model = builder(input_size=input_size, num_classes=y_train.shape[1])

    ckpt = os.path.join(OUT_DIR, f'best_{name}.keras')
    callbacks = [EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
                 ModelCheckpoint(ckpt, save_best_only=True, monitor='val_loss', mode='min')]

    t0 = time.perf_counter()
    history = model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=MAX_EPOCHS,
                        validation_data=(X_val, y_val),   # <-- validation split only
                        callbacks=callbacks, verbose=verbose)
    train_time = time.perf_counter() - t0

    # Final metrics on the held-out TEST set.
    y_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)

    infer_ms = measure_inference_time(model, X_test)
    params = model.count_params()
    size_mb = os.path.getsize(ckpt) / 1e6 if os.path.exists(ckpt) else float('nan')

    row = {
        'Model Name': name,
        'Accuracy': round(acc * 100, 2),
        'Precision': round(prec * 100, 2),
        'Recall': round(rec * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'Training Time': format_seconds(train_time),
        'Inference Time (ms)': round(infer_ms, 3),
        'Total Parameters': int(params),
        'Model Size (MB)': round(size_mb, 2),
        'Epochs': len(history.history['loss']),
    }
    print({k: row[k] for k in ['Accuracy', 'Precision', 'Recall',
                               'Inference Time (ms)', 'Total Parameters', 'Model Size (MB)']})
    # Free memory before the next model.
    del model; keras.backend.clear_session(); gc.collect()
    return row, history


def run_benchmark(model_names, X_train, y_train, X_val, y_val, X_test, y_test,
                  input_size=128, verbose=0):
    rows = []
    for name in model_names:
        row, _ = benchmark_model(name, MODEL_REGISTRY[name],
                                 X_train, y_train, X_val, y_val, X_test, y_test,
                                 input_size=input_size, verbose=verbose)
        rows.append(row)
    return pd.DataFrame(rows)


## 6. Table II — all models at 128×128

Builds the 3-way split once at 128×128, then benchmarks every registered model. Each model is **trained** on the training split, **early-stopped** on the validation split, and **scored on the held-out test set**.

> Training 17 models (several large pretrained backbones) is time-consuming and needs a GPU. To benchmark a subset, edit `models_to_run`.

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = prepare_data(img_size=128)


In [ ]:
models_to_run = TABLE2_ORDER          # or e.g. list(SIMPLENET_CONFIGS) for the from-scratch models only
table2 = run_benchmark(models_to_run, X_train, y_train, X_val, y_val, X_test, y_test,
                       input_size=128, verbose=0)

# run_benchmark already returns rows in the requested order.
table2_path = os.path.join(OUT_DIR, 'table2_128x128.csv')
table2.to_csv(table2_path, index=False)
print('Saved ->', table2_path)
table2

## 7. Table IV — best models at 64×64

Re-runs the two retained models (**SimpleNetAugDR-3**, **MobileNet2**) on 64×64 images, using the same train / validation / held-out-test protocol.

In [ ]:
X_train64, y_train64, X_val64, y_val64, X_test64, y_test64 = prepare_data(img_size=64)

table4 = run_benchmark(TABLE4_ORDER, X_train64, y_train64, X_val64, y_val64, X_test64, y_test64,
                       input_size=64, verbose=0)
table4_path = os.path.join(OUT_DIR, 'table4_64x64.csv')
table4.to_csv(table4_path, index=False)
print('Saved ->', table4_path)
table4

## 8. (Optional) Qualitative evaluation of a single model

Retrains one model with `verbose=1` and inspects its training curves, plus its confusion matrix and ROC curve **on the held-out test set**.

In [ ]:
MODEL_TO_INSPECT = 'SimpleNetAugDR-3'
row, history = benchmark_model(MODEL_TO_INSPECT, MODEL_REGISTRY[MODEL_TO_INSPECT],
                               X_train, y_train, X_val, y_val, X_test, y_test,
                               input_size=128, verbose=1)

best = keras.models.load_model(os.path.join(OUT_DIR, f'best_{MODEL_TO_INSPECT}.keras'))
plot_training_metrics(history)

# Confusion matrix + ROC on the held-out TEST set.
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(best.predict(X_test, verbose=0), axis=1)
plot_confusion_matrix(confusion_matrix(y_true, y_pred), classes=CLASS_NAMES, normalize=True)
plot_roc_curve(best, X_test, y_test)